In [ ]:
import os
os.environ["GOOGLE_API_KEY"]="abc"

In [2]:
pip install langchain-text-splitters youtube-transcript-api langchain-google-genai langchain-community langchain-core chromadb faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install langchain-chroma

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
Using cached langchain_chroma-1.1.0-py3-none-any.whl (12 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from youtube_transcript_api import YouTubeTranscriptApi


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma, FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [ ]:
#Step 1a Indexing (Document Ingestion)o

In [ ]:
# {
#     "text": "Hello everyone",
#     "start": 0.0,
#     "duration": 2.5
# }

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
video_id = "P26AE7NLx4Q"
text=""
try:
    api = YouTubeTranscriptApi()
    transcript = api.fetch(video_id,languages=["en"])
    text = " ".join(chunk.text for chunk in transcript)
    print(text)
except Exception as e:
    print("Transcript not available:", e)

Transcript not available: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=P26AE7NLx4Q! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you have u

In [ ]:
# Step 1b Indexing (Text Splitting)



In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.create_documents([text])

In [ ]:
len(chunks)

22

In [ ]:
chunks[1]

Document(metadata={}, page_content="the ultimate culmination  of Ormund's gambit, and I think he's played it\nto the very end. Ulf hasn't reported, and he's not with his dragon. RYAN: Now the Rivermen\nhave shown up at their gates,  and they're gonna go in.  This is a good, proper,\n old medieval siege.  ANDRIJ PAREKH:\n The art department\n kindly sent me a 1/100th scale model\nof Tumbleton. VANJA: First thing we did,\nwe bought little figurines, and we were like playing\nlike two kids with this model of Tumbleton,  and we're planning\n how we want to approach  each particular scene. RYAN: It was months of crafting\n storyboards and previs  and showing it to me, and we very much assembled\nthe battle in prep  and were able to plan\n and shift things  based on how we were\n seeing it come together. Jim Clay and his team built this\nincredible set on our backlot. We had an incredible location,\nHankley Common in Surrey,  to marry it to. DOMINIC MASTERS:\nJim's great idea of putting\n th

In [ ]:
print(chunks[1].page_content)

the ultimate culmination  of Ormund's gambit, and I think he's played it
to the very end. Ulf hasn't reported, and he's not with his dragon. RYAN: Now the Rivermen
have shown up at their gates,  and they're gonna go in.  This is a good, proper,
 old medieval siege.  ANDRIJ PAREKH:
 The art department
 kindly sent me a 1/100th scale model
of Tumbleton. VANJA: First thing we did,
we bought little figurines, and we were like playing
like two kids with this model of Tumbleton,  and we're planning
 how we want to approach  each particular scene. RYAN: It was months of crafting
 storyboards and previs  and showing it to me, and we very much assembled
the battle in prep  and were able to plan
 and shift things  based on how we were
 seeing it come together. Jim Clay and his team built this
incredible set on our backlot. We had an incredible location,
Hankley Common in Surrey,  to marry it to. DOMINIC MASTERS:
Jim's great idea of putting
 these farmhouses there really locked everything


In [ ]:
print(len(text))

17135


In [ ]:
# Step 1c and 1d (Embedding Generation and Storing in vector store)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
from langchain_chroma import Chroma

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="my_chroma_db",
    collection_name="sample"
)

/tmp/ipykernel_1048/3363286737.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [ ]:
# add documents
vector_store.add_documents(chunks)

['74d9e859-a8f8-4d07-af4f-a44d12eed127',
 'b4f446db-ac91-42a4-ab40-6cd249617a8a',
 '8d731836-7901-47a1-a303-d936b1c99e43',
 'f66e49cb-f9ff-4eff-8040-f8371ca8d042',
 'ed8867be-c19f-43c7-8df8-e53d1b5731eb',
 '44d22d58-7653-478f-871e-f55820c841c0',
 'b9e7d6cf-4e11-49db-a693-7e6b58cc46a9',
 '3841ff8d-d400-46e2-a8c8-2ace60131a12',
 '863fc3fc-1665-4924-8c77-53b3998d0b8a',
 'ed3ea038-077c-40a5-bc10-7a034db010eb',
 '27c34481-7cc6-4d6a-bc3c-6a490f15304a',
 'febed8a8-0dd2-4fdd-97a2-d05c7cdb5c3b',
 '4b66b038-ee95-4a02-b604-e0c193da3511',
 '4d9c32ef-066b-439d-a8e5-f823bca9d374',
 '948f5c2a-b258-4ce9-b77c-a249490a2781',
 '1100cb45-ac36-461c-9c9e-1cfeb01fdac6',
 '721d76d7-a871-4cd4-bc7f-948dea43da58',
 'df40a54a-9d04-4674-a8f8-77b0f28c7b5d',
 '30a2e8a0-cadc-4f05-8108-95f2335ec586',
 'd2f60ddd-66a0-45c3-a694-7f2a06938372',
 '7bb1273c-33a2-44e3-86e8-4959c56596af',
 '3cf54282-2614-4b39-a437-3bbee35f1956']

In [ ]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['74d9e859-a8f8-4d07-af4f-a44d12eed127',
  'b4f446db-ac91-42a4-ab40-6cd249617a8a',
  '8d731836-7901-47a1-a303-d936b1c99e43',
  'f66e49cb-f9ff-4eff-8040-f8371ca8d042',
  'ed8867be-c19f-43c7-8df8-e53d1b5731eb',
  '44d22d58-7653-478f-871e-f55820c841c0',
  'b9e7d6cf-4e11-49db-a693-7e6b58cc46a9',
  '3841ff8d-d400-46e2-a8c8-2ace60131a12',
  '863fc3fc-1665-4924-8c77-53b3998d0b8a',
  'ed3ea038-077c-40a5-bc10-7a034db010eb',
  '27c34481-7cc6-4d6a-bc3c-6a490f15304a',
  'febed8a8-0dd2-4fdd-97a2-d05c7cdb5c3b',
  '4b66b038-ee95-4a02-b604-e0c193da3511',
  '4d9c32ef-066b-439d-a8e5-f823bca9d374',
  '948f5c2a-b258-4ce9-b77c-a249490a2781',
  '1100cb45-ac36-461c-9c9e-1cfeb01fdac6',
  '721d76d7-a871-4cd4-bc7f-948dea43da58',
  'df40a54a-9d04-4674-a8f8-77b0f28c7b5d',
  '30a2e8a0-cadc-4f05-8108-95f2335ec586',
  'd2f60ddd-66a0-45c3-a694-7f2a06938372',
  '7bb1273c-33a2-44e3-86e8-4959c56596af',
  '3cf54282-2614-4b39-a437-3bbee35f1956'],
 'embeddings': array([[ 0.00939054, -0.01469993,  0.01552714, ..., -

In [ ]:
vector_store.get(
    ids=["74d9e859-a8f8-4d07-af4f-a44d12eed127"],
    include=["documents", "metadatas"]
)

{'ids': ['74d9e859-a8f8-4d07-af4f-a44d12eed127'],
 'embeddings': None,
 'documents': ['VANJA CERNJUL: It was sometime,\n I think, in November, Andrij called me and asked me, "Do you want to come back\nand do something spectacular?" And I was like,\n"Yes, of course." I spent season two with Vanja,  and what he does\nas a cinematographer is amazing. It was really special\nto have Vanja back. RYAN CONDAL: This is our finale. We wanted to go out with a bang.  In many ways, Tumbleton is an indicator\nof the things to come in The Dance of the Dragons. ROWLEY IRLAM: All right,\nhere we go now! Ready? And three, two, one, action! ♪ (THRILLING MUSIC PLAYING) ♪ ♪ (MUSIC ENDS) ♪ We can make...\nWe can make their start line... CREW MEMBER:\nAll right, so you jump\non the stake mark there, please. Tommy will walk\nto there with the guys. They\'ll all take\nlike four steps. Yeah. ♪ (TENSE MUSIC PLAYING) ♪ RYAN: I mean, this is\n the ultimate culmination  of Ormund\'s gambit, and I think he\'s played

In [ ]:
# Step 2 Retrieval

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)

In [ ]:
retriever

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7d732c761c70>, search_kwargs={'k': 1})

In [ ]:
# Search
# q="How was the Tumbleton battle filmed?"
query = "How was Ormund Hightower's death filmed?"
docs = retriever.invoke(query)

for doc in docs:
    print(doc.page_content)

sword of his own  called Orphan-Maker. (GRUNTING) As fans, we wanted
to see that fight.  Two seasoned warriors,
two Valyrian steel swords,  two of our favorites
 having a go. (GRUNTING) Well, I was very keen that
that wasn't just a sword fight. I wanted it to be a punch-up,
 a violent punch-up, like a bar brawl,
quick, unexpected. There's a real adrenaline to it,
and I'd take that  over being sat
 in a council chamber.  But I know you need
the council chamber stuff for the context. (GRUNTING) JAMES NORTON:
 There were many, many
 very memorable moments for me on this job. I think Ormund's death
will go down  as one of my all-time
 greatest deaths. Rowley, our stunt coordinator, had a genius idea
of how to spice it up, and I don't think
this death disappoints. ROWLEY:
 So they have this fight. (GRUNTING) And then at the end,  there's this line. That's a proper fucking... by the North! ROWLEY: And I was like, "Okay." So he's bent over


In [ ]:
# Step 3 Augmentation

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
)


In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

{context}
Question: {question}
""",
    input_variables=['context', 'question']
)

In [ ]:
question = "What happened during the siege of Tumbleton?"

retrieved_docs = retriever.invoke(question)

In [ ]:
context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

In [ ]:
final_prompt = prompt.invoke({
    "context": context,
    "question": question
})

print(final_prompt)

text='\nYou are a helpful assistant.\nAnswer ONLY from the provided transcript context.\nIf the context is insufficient, just say you don\'t know.\n\nit was just before lunch,  and everyone was\n getting really hungry,  and it was like,\n "Come on, hurry up," so. ROWLEY: As far away from ladder\nas possible, and then literally, Bradley can watch that camera,\nand as Tommy\'s halfway, just start climbing, and he\'ll be at the top\nof the ladder when Tommy\'s... -Start there.\n-Yeah, just give us a-- Peel back. Just rile up the troops again. Everybody ready?\nAnd three, two, one, action! (ALL YELLING) ♪ (THRILLING MUSIC PLAYING) ♪ ANDRIJ: So this episode required a tremendous amount\nof stunt work. I think at one point we had  probably all\n of London\'s stunt guys all working on the same show\n at the same time. -ROWLEY: Action!\n-(ALL YELLING) So we got\nthe siege of Tumbleton. We have done sieges before,\nnot on this scale though.  This is big scale. A lot of the A side of that\nwas d

In [ ]:
# Step 4 Generation

In [ ]:
result=model.invoke(final_prompt)
print(result.content[0]["text"])

Based on the provided transcript, the text does not detail the plot or in-universe events of the siege of Tumbleton. It only mentions the behind-the-scenes production details:

* It was filmed on a "big scale."
* It required a tremendous amount of stunt work, utilizing probably all of London's stunt crew at the same time.
* A lot of the "A side" was filmed at Hankley Common.
* The rest was filmed at the studio on a huge backlot set built by Jim Clay and his team.
